In [134]:
import string
from unittest.mock import inplace

import pandas as pd
import numpy as np
from nltk.corpus import stopwords
from pygments.styles.dracula import background
from sklearn.conftest import pytest_generate_tests
from sympy import vectorize

In [135]:
data=pd.read_csv('spam.csv',encoding='latin-1')
data.sample(10)

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
4611,ham,Sorry da. I gone mad so many pending works wha...,NaN,NaN,NaN
506,ham,"Maybe westshore or hyde park village, the plac...",NaN,NaN,NaN
4982,ham,Networking job is there.,NaN,NaN,NaN
3430,ham,Yeah if we do have to get a random dude we nee...,NaN,NaN,NaN
4806,spam,PRIVATE! Your 2004 Account Statement for 07849...,NaN,NaN,NaN
357,spam,Ur cash-balance is currently 500 pounds - to m...,NaN,NaN,NaN
2133,ham,Spoke with uncle john today. He strongly feels...,NaN,NaN,NaN
5225,ham,Smile in Pleasure Smile in Pain Smile when tro...,NaN,NaN,NaN
3143,ham,"Haha I heard that, text me when you're around",NaN,NaN,NaN
5223,ham,If I die I want u to have all my stuffs.,NaN,NaN,NaN


In [136]:
data.info()
data=data.drop(columns=['Unnamed: 2','Unnamed: 3','Unnamed: 4'])
data.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   v1          5572 non-null   object
 1   v2          5572 non-null   object
 2   Unnamed: 2  50 non-null     object
 3   Unnamed: 3  12 non-null     object
 4   Unnamed: 4  6 non-null      object
dtypes: object(5)
memory usage: 217.8+ KB


,v1,v2
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [137]:
data.rename(columns={'v1':'target','v2':'text'},inplace=True)

In [138]:
data['target'].unique()

array(['ham', 'spam'], dtype=object)

In [139]:
data['target']=data['target'].map({'ham':0,'spam':1})

In [140]:
data.head()

,target,text
0,0,"Go until jurong point, crazy.. Available only ..."
1,0,Ok lar... Joking wif u oni...
2,1,Free entry in 2 a wkly comp to win FA Cup fina...
3,0,U dun say so early hor... U c already then say...
4,0,"Nah I don't think he goes to usf, he lives aro..."


In [141]:
import random

subjects = [
    "Congratulations", "URGENT", "Dear user", "Alert", "Limited time offer",
    "Exclusive deal", "Important notice", "Final reminder"
]

actions = [
    "you have won", "you are selected for", "claim your", "get your",
    "verify your", "update your", "unlock your", "activate your"
]

objects = [
    "lottery prize", "free car", "cash reward", "iPhone", "gift voucher",
    "bank account", "KYC details", "OTP verification", "delivery package"
]

values = [
    "₹10,000", "₹50,000", "₹1 lakh", "₹25 lakh", "free", "exclusive"
]

endings = [
    "now", "immediately", "today", "before expiry", "limited time",
    "click here", "act fast", "don’t miss this"
]

templates = [
    "{s}! {a} a {v} {o}. {e}",
    "{s}: {a} your {o} {e}",
    "{s}! {a} {o}. {e}",
    "{s}, {a} {v} {o} {e}",
]

def generate_spam(n=1000):
    spam_list = []
    for _ in range(n):
        msg = random.choice(templates).format(
            s=random.choice(subjects),
            a=random.choice(actions),
            v=random.choice(values),
            o=random.choice(objects),
            e=random.choice(endings)
        )
        spam_list.append(msg)
    return spam_list

spam_messages = generate_spam(1000)
spam_messages

['Exclusive deal: update your your cash reward immediately',
 'Limited time offer! activate your lottery prize. don’t miss this',
 'Limited time offer! activate your cash reward. before expiry',
 'Alert: you are selected for your KYC details act fast',
 'Congratulations! update your lottery prize. today',
 'Dear user! you have won gift voucher. click here',
 'URGENT! unlock your a ₹25 lakh gift voucher. immediately',
 'Congratulations, you are selected for ₹10,000 delivery package immediately',
 'Alert, you have won ₹25 lakh gift voucher don’t miss this',
 'Important notice: activate your your delivery package before expiry',
 'Congratulations! you have won a ₹1 lakh free car. act fast',
 'Important notice! unlock your a free lottery prize. before expiry',
 'Exclusive deal: you have won your delivery package limited time',
 'Congratulations: update your your gift voucher immediately',
 'Dear user! unlock your a ₹10,000 lottery prize. limited time',
 'Important notice! unlock your a exc

In [142]:
new_df=pd.DataFrame({
    'text':spam_messages,
    'target':[1]*len(spam_messages)
})
new_df

,text,target
0,Exclusive deal: update your your cash reward i...,1
1,Limited time offer! activate your lottery priz...,1
2,Limited time offer! activate your cash reward....,1
3,Alert: you are selected for your KYC details a...,1
4,Congratulations! update your lottery prize. today,1
...,...,...
995,Limited time offer! you have won OTP verificat...,1
996,Exclusive deal! update your a exclusive iPhone...,1
997,Exclusive deal! claim your a ₹25 lakh OTP veri...,1
998,Alert! get your iPhone. before expiry,1


In [143]:
data=pd.concat([data,new_df],ignore_index=True)
data=data.sample(frac=1,random_state=42).reset_index(drop=True)
data.tail(10)

,target,text
6562,0,"Garbage bags, eggs, jam, bread, hannaford whea..."
6563,0,They don't put that stuff on the roads to keep...
6564,1,Final reminder: you have won your lottery priz...
6565,1,Dear user! you have won gift voucher. today
6566,0,staff.science.nus.edu.sg/~phyhcmk/teaching/pc1323
6567,0,I came hostel. I m going to sleep. Plz call me...
6568,0,"Sorry, I'll call later"
6569,0,Prabha..i'm soryda..realy..frm heart i'm sory
6570,0,Nt joking seriously i told
6571,0,In work now. Going have in few min.


In [144]:
data.isna().sum()

target    0
text      0
dtype: int64

In [145]:
#Checking duplicates
data.duplicated().sum()

np.int64(420)

In [146]:
data=data.drop_duplicates(keep='first')
data.shape

(6152, 2)

In [147]:
#EDA
!pip install matplotlib



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [148]:
import matplotlib
matplotlib.use('TkAgg')
import matplotlib.pyplot as plt
# cnt=data['target'].value_counts()
# percentage=(cnt/(len(cnt)))*100
# plot=data['target'].value_counts(normalize=True).plot(kind='bar')
# for ind,val in enumerate(percentage):
#     plot.text(ind,cnt.values[ind]+0.1,f"{val:.2f}%",ha='center')
# plt.show()
cnt = data['target'].value_counts()
percentage = cnt / len(data) * 100

ax = cnt.plot(kind='bar')

for i, v in enumerate(cnt):
    ax.text(i, v + 0.1, f"{percentage.iloc[i]:.2f}%", ha='center')

plt.title("Target Distribution")
plt.xlabel("Target")
plt.ylabel("Count")

plt.show()


In [149]:
import matplotlib.pyplot as plt
plt.pie(cnt,labels=['not spam','spam'],autopct='%0.2f')
plt.show()

In [150]:
!pip install nltk



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [151]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\palla\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\palla\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\palla\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [152]:
data['num_char']=data['text'].apply(len)
data.head()

,target,text,num_char
0,0,Its a valentine game. . . Send dis msg to all ...,156
1,0,"I know where the &lt;#&gt; is, I'll be there...",55
2,0,NEFT Transaction with reference number &lt;#&...,164
3,1,"Alert, get your free gift voucher click here",44
4,0,Easy ah?sen got selected means its good..,41


In [153]:
data['num_words']=data['text'].apply(lambda x:len(nltk.word_tokenize(x)))
data.head()

,target,text,num_char,num_words
0,0,Its a valentine game. . . Send dis msg to all ...,156,37
1,0,"I know where the &lt;#&gt; is, I'll be there...",55,19
2,0,NEFT Transaction with reference number &lt;#&...,164,53
3,1,"Alert, get your free gift voucher click here",44,9
4,0,Easy ah?sen got selected means its good..,41,10


In [154]:
data['num_sen']=data['text'].apply(lambda x:len(nltk.sent_tokenize(x)))
data.head()

,target,text,num_char,num_words,num_sen
0,0,Its a valentine game. . . Send dis msg to all ...,156,37,6
1,0,"I know where the &lt;#&gt; is, I'll be there...",55,19,1
2,0,NEFT Transaction with reference number &lt;#&...,164,53,2
3,1,"Alert, get your free gift voucher click here",44,9,1
4,0,Easy ah?sen got selected means its good..,41,10,1


In [155]:
#For not spam
data[data['target']==0][['num_char','num_words','num_sen']].describe()

,num_char,num_words,num_sen
count,4516.000000,4516.000000,4516.000000
mean,70.459256,17.123782,1.820195
std,56.358207,13.493970,1.383657
min,2.000000,1.000000,1.000000
25%,34.000000,8.000000,1.000000
50%,52.000000,13.000000,1.000000
75%,90.000000,22.000000,2.000000
max,910.000000,220.000000,38.000000


In [156]:
#For spam
data[data['target']==1][['num_char','num_words','num_sen']].describe()

,num_char,num_words,num_sen
count,1636.000000,1636.000000,1636.000000
mean,88.613692,17.438264,2.382029
std,44.942082,9.551367,1.309628
min,13.000000,2.000000,1.000000
25%,54.000000,10.000000,1.000000
50%,63.000000,12.000000,3.000000
75%,141.000000,27.000000,3.000000
max,224.000000,46.000000,9.000000


In [157]:
!pip install seaborn


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [158]:
import seaborn as sns
plt.figure(figsize=(12,6))
sns.histplot(data[data['target']==0]['num_words'],color='green')
sns.histplot(data[data['target']==1]['num_words'],color='red')

plt.show()

In [159]:
plt.figure(figsize=(12,6))
sns.histplot(data[data['target']==0]['num_char'],color='green')
sns.histplot(data[data['target']==1]['num_char'],color='red')
plt.show()

In [160]:
plt.figure(figsize=(12,6))
sns.histplot(data[data['target']==0]['num_sen'],color='green')
sns.histplot(data[data['target']==1]['num_sen'],color='red')
plt.show()

In [161]:
sns.heatmap(data[['target','num_char','num_words','num_sen']].corr(),annot=True)
plt.show()

Text Preprocessing

In [162]:
from nltk.stem.porter import PorterStemmer
ps=PorterStemmer()

In [163]:
import string
def text_tranform(text):
    text=text.lower()
    text=nltk.word_tokenize(text)
    res=[]
    for i in text:
        if i.isalnum():
            res.append(i)
    text=res[:]
    res.clear()
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            res.append(i)
    text=res[:]
    # res.clear()
    # for i in text:
    #     res.append(ps.stem(i))
    return ' '.join(res)

In [164]:
data['transformed_text']=data['text'].apply(text_tranform)

In [165]:
data.head()

,target,text,num_char,num_words,num_sen,transformed_text
0,0,Its a valentine game. . . Send dis msg to all ...,156,37,6,valentine game send dis msg ur friends 5 answe...
1,0,"I know where the &lt;#&gt; is, I'll be there...",55,19,1,know lt gt around 5
2,0,NEFT Transaction with reference number &lt;#&...,164,53,2,neft transaction reference number lt gt rs lt ...
3,1,"Alert, get your free gift voucher click here",44,9,1,alert get free gift voucher click
4,0,Easy ah?sen got selected means its good..,41,10,1,easy ah sen got selected means good


In [166]:
!pip install wordcloud


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [167]:
from wordcloud import WordCloud
wc=WordCloud(width=1000,height=1000,background_color='beige')


In [168]:
spam_wc=wc.generate(data[data['target']==0]['transformed_text'].str.cat(sep=" "))

In [169]:
plt.imshow(spam_wc)
plt.show()

In [170]:
ham_wc=wc.generate(data[data['target']==1]['transformed_text'].str.cat(sep=" "))

In [171]:
plt.imshow(ham_wc)
plt.show()

In [172]:
from collections import Counter
ham_corpus=[]
for sen in data[data['target']==0]['transformed_text'].to_list():
    for word in sen.split():
        ham_corpus.append(word)
word_count=Counter(ham_corpus).most_common(30)
sns.barplot(x=pd.DataFrame(word_count)[0],y=pd.DataFrame(word_count)[1],palette='magma',legend=False,hue=pd.DataFrame(word_count)[0])
plt.xticks(rotation='vertical')
plt.show()

In [173]:
spam_corpus=[]
for sen in data[data['target']==1]['transformed_text'].to_list():
    for word in sen.split():
        spam_corpus.append(word)
word_count=Counter(spam_corpus).most_common(30)
sns.barplot(x=pd.DataFrame(word_count)[0],y=pd.DataFrame(word_count)[1],palette='flare',legend=False,hue=pd.DataFrame(word_count)[0])
plt.xticks(rotation='vertical')
plt.show()

Model

In [174]:
print(text_tranform('congratulation you won a car in lucky draw'))

congratulation car lucky draw


In [187]:
#Using bag of words vectorization method
from sklearn.feature_extraction.text import CountVectorizer,TfidfVectorizer
cv=CountVectorizer()
vectorizer=TfidfVectorizer(ngram_range=(1,2),max_features=3000)

In [188]:
X=vectorizer.fit_transform(data['transformed_text']).toarray()
X.shape


(6152, 3000)

In [189]:
features = vectorizer.get_feature_names_out()

print("lucky draw" in features)  # should be True
print("won car" in features)    # should be True

False
False


In [190]:
Y=data['target'].values
Y

array([0, 0, 0, ..., 0, 0, 0], shape=(6152,))

In [191]:
from sklearn.model_selection import train_test_split
X_train,X_test,Y_train,Y_test=train_test_split(X,Y,test_size=0.2,random_state=2)

In [192]:
from sklearn.naive_bayes import GaussianNB,MultinomialNB,BernoulliNB
from sklearn.metrics import accuracy_score,precision_score,confusion_matrix
gn=GaussianNB()
mn=MultinomialNB()
bn=BernoulliNB()


In [193]:
gn.fit(X_train,Y_train)
y_pred1=gn.predict(X_test)
print(accuracy_score(Y_test,y_pred1))
print(precision_score(Y_test,y_pred1))
print(confusion_matrix(Y_test,y_pred1))


0.9073923639317628
0.7790697674418605
[[782  95]
 [ 19 335]]


In [194]:
mn.fit(X_train,Y_train)
y_pred2=mn.predict(X_test)
print(accuracy_score(Y_test,y_pred2))
print(precision_score(Y_test,y_pred2))
print(confusion_matrix(Y_test,y_pred2))


0.9723801787164906
1.0
[[877   0]
 [ 34 320]]


In [195]:
bn.fit(X_train,Y_train)
y_pred3=bn.predict(X_test)
print(accuracy_score(Y_test,y_pred3))
print(precision_score(Y_test,y_pred3))
print(confusion_matrix(Y_test,y_pred3))


0.9780666125101544
1.0
[[877   0]
 [ 27 327]]


In [196]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

In [197]:
models = {
    "Logistic Regression": LogisticRegression(),
    "Gaussian NB": GaussianNB(),
    "Multinomial NB": MultinomialNB(),
    "Bernoulli NB": BernoulliNB(),
    "Decision Tree": DecisionTreeClassifier(),
    "Random Forest": RandomForestClassifier(),
    "Extra Trees": ExtraTreesClassifier(),
    "AdaBoost": AdaBoostClassifier(),
    "Gradient Boosting": GradientBoostingClassifier(),
    "KNN": KNeighborsClassifier(),
    "SVC": SVC()
}
def train_model(model, X_train, Y_train, X_test, Y_test):

    model.fit(X_train, Y_train)

    y_pred = model.predict(X_test)

    acc = accuracy_score(Y_test, y_pred)
    prec = precision_score(Y_test, y_pred)

    return acc, prec
accuracy_scores = []
precision_scores = []
model_names = []

for name, model in models.items():

    acc, prec = train_model(model, X_train, Y_train, X_test, Y_test)

    model_names.append(name)
    accuracy_scores.append(acc)
    precision_scores.append(prec)
    results = pd.DataFrame({
    "Model": model_names,
    "Accuracy": accuracy_scores,
    "Precision": precision_scores
})

results.sort_values(by="Precision", ascending=False)

,Model,Accuracy,Precision
2,Multinomial NB,0.972380,1.000000
3,Bernoulli NB,0.978067,1.000000
0,Logistic Regression,0.965881,0.996815
9,KNN,0.919578,0.996109
6,Extra Trees,0.978067,0.993958
10,SVC,0.973193,0.993846
8,Gradient Boosting,0.959383,0.993506
5,Random Forest,0.972380,0.987805
4,Decision Tree,0.949634,0.926901
7,AdaBoost,0.808286,0.898649


In [199]:
best_model=models['Logistic Regression']
best_model.fit(X_train,Y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [200]:
import pickle
pickle.dump(vectorizer,open('vectorizer.pkl','wb'))
pickle.dump(best_model,open('model.pkl','wb'))